# 02 · RAG & Embeddings

> **Source notes:** `RAGAndEmbeddings.md` + `RAGAndEmbeddings_Supplement.md`

How does an agent retrieve knowledge it wasn't trained on?

This notebook covers **embeddings end-to-end**: creation, pooling strategies, the full RAG ingestion and query pipeline, advanced patterns (HyDE), and production failure modes.

**Tools (all local, no API key needed):**
- `sentence-transformers` — embed text with a local model
- `ollama` — local SLM for generation
- `chromadb` — local in-memory vector store

In [ ]:
## 0 · Setup
import subprocess, sys
pkgs = ["sentence-transformers", "ollama", "chromadb", "scikit-learn", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("Packages ready.")
# Reminder: make sure `ollama serve` is running and you have pulled a model:
#   ollama pull phi3:mini

## 1 · What Are Embeddings?

**Embeddings** transform text into fixed-size numerical vectors where *meaning becomes measurable*: semantically similar text clusters together in high-dimensional space.

### How a Transformer Encoder Produces an Embedding

```
Input text
  --> Tokenise  (split into word-pieces, add [CLS] and [SEP])
  --> Token + Position Embeddings
  --> Self-Attention x N layers   (every token attends to every other token -- O(n^2))
  --> Final hidden states  [N_tokens x 768]
  --> Pooling  (collapse to a single vector)
  --> Embedding  [768]
```

Key contrast: **encoder** models (BERT, all-MiniLM) process the entire input at once.  
**Decoder** models (GPT, Llama, Phi) process left-to-right and are used for generation.

### Pooling Strategies

| Strategy | Mechanism | When Used |
|----------|-----------|-----------|
| **[CLS] token** | Final hidden state of the special first token | BERT-family models |
| **Mean pooling** | Average all token embeddings | Most modern embedding models |
| **Max pooling** | Element-wise max across tokens | Specialised retrieval |
| **Last-token** | Final non-padding token hidden state | Decoder-based embeddings |

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# all-MiniLM-L6-v2: 23 MB, 384-dimensional, very fast
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "A train travels from Seattle to Vancouver in 4 hours.",          # ref
    "The rail journey from SEA to YVR covers 230 kilometres.",        # closely related
    "A locomotive averages 57.5 km/h on the Pacific Northwest route.",# related
    "The speed of light is approximately 299,792 km/s.",              # unrelated
]

# normalize_embeddings=True makes cosine similarity = dot product
embeddings = embed_model.encode(sentences, normalize_embeddings=True)
print(f"Embedding shape: {embeddings.shape}  ({len(sentences)} texts x {embeddings.shape[1]} dims)\n")

# Cosine similarity via dot product (vectors are already normalised)
REF = 0
print(f"Reference: '{sentences[REF]}'\n")
for i, s in enumerate(sentences):
    sim = float(np.dot(embeddings[REF], embeddings[i]))
    bar = "###" * int(sim * 10)
    print(f"  {sim:.3f}  {bar:<30}  '{s[:55]}'")

In [ ]:
# Visualise the embedding space in 2D with PCA
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

labels = ["SEA-YVR 4h (ref)", "SEA-YVR 230km", "57.5 km/h loco", "Speed of light"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(coords[:, 0], coords[:, 1], s=120, zorder=3)
for (x, y), lbl in zip(coords, labels):
    ax.annotate(lbl, (x, y), textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_title("Embedding space — PCA 2D projection")
ax.set_xlabel("PC-1"); ax.set_ylabel("PC-2")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Rail-related sentences cluster together; 'speed of light' is far apart.")

## 2 · Contrastive Learning — How Embedding Models Are Trained

Embedding models are **not** trained to predict the next token.  
They use **contrastive learning**: push similar texts together, push unrelated texts apart.

The most common objective is **InfoNCE loss**:

$$
\mathcal{L} = -\log \frac{\exp(\text{sim}(q, p^+) / \tau)}{\exp(\text{sim}(q, p^+) / \tau) + \sum_i \exp(\text{sim}(q, p_i^-) / \tau)}
$$

- $q$ = query embedding, $p^+$ = positive (similar), $p_i^-$ = negatives
- $\tau$ = temperature (controls how "peaked" the distribution is)

In practice, positive pairs come from: (question, answer), (sentence, paraphrase), (chunk, its document title).

## 3 · The RAG Pipeline

RAG (Retrieval-Augmented Generation) gives the LLM access to a private or up-to-date corpus:

```
INGESTION (offline)                     QUERY (runtime)
-------------------                     ---------------
Documents                               User question
  --> chunk                               --> embed question
  --> embed each chunk                    --> ANN search --> top-k chunks
  --> store in vector DB                  --> build prompt (context + question)
                                          --> LLM generates grounded answer
```

Two phases:
1. **Ingestion** — split documents into chunks, embed, store
2. **Query** — embed the question, retrieve similar chunks, augment the LLM prompt

In [ ]:
import chromadb, ollama

MODEL = "phi3:mini"   # change to your pulled model

# === INGESTION ===
DOCS = [
    "The SEA-YVR rail route covers approximately 230 kilometres.",
    "The Amtrak Cascades connects Seattle (SEA) and Vancouver (YVR).",
    "Average speed on the SEA-YVR corridor is about 57.5 km/h.",
    "The Amtrak Cascades is a Talgo Series 8 trainset with a top speed of 200 km/h.",
    "Journey time between Seattle and Vancouver is roughly 4 hours by train.",
    "The Cascades corridor passes through Tacoma, Olympia, and Bellingham.",
]

client = chromadb.Client()   # in-memory, no files written
col = client.get_or_create_collection("rail-docs")

vecs = embed_model.encode(DOCS, normalize_embeddings=True).tolist()
col.add(ids=[f"doc{i}" for i in range(len(DOCS))], embeddings=vecs, documents=DOCS)
print(f"Ingested {len(DOCS)} documents.\n")

# === QUERY ===
QUESTION = "What is the maximum speed of the train on the SEA-YVR route?"
q_vec = embed_model.encode([QUESTION], normalize_embeddings=True).tolist()
results = col.query(query_embeddings=q_vec, n_results=3)
retrieved = results["documents"][0]

print("Retrieved chunks:")
for i, chunk in enumerate(retrieved, 1):
    print(f"  {i}. {chunk}")

# === AUGMENTED GENERATION ===
context = "\n".join(f"- {c}" for c in retrieved)
prompt = (
    "Answer the question using ONLY the provided context.\n"
    "If the context lacks information, say so.\n\n"
    f"Context:\n{context}\n\nQuestion: {QUESTION}\nAnswer:"
)
resp = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                   options={"temperature": 0.0})
print(f"\nAnswer: {resp['message']['content']}")

## 4 · Advanced RAG — HyDE (Hypothetical Document Embeddings)

**Problem:** A user question (*"What is the average rail speed?"*) and its answer document (*"The average speed is 57.5 km/h..."*) are phrased differently. This **semantic gap** degrades retrieval.

**HyDE solution:**
1. Ask the LLM to generate a *hypothetical answer document* (even if some details are approximate)
2. Embed **that hypothetical document** instead of the raw question
3. Search with the hypothetical embedding — it matches real documents much better

```
User query  --(LLM)-->  Hypothetical doc  --(embed)-->  vector  --(search)-->  real docs
                         (plausible answer)
```

This works because the generated answer has the same **phrasing style** as the real answer documents.

### Other Advanced Patterns

| Pattern | What It Does |
|---------|-------------|
| **FLARE** | Re-retrieves mid-generation when confidence drops |
| **Query Decomposition** | Breaks complex queries into sub-queries, retrieves for each |
| **Re-ranking** | Cross-encoder re-scores retrieved chunks for precision |
| **Sparse + Dense (Hybrid)** | BM25 keyword search combined with dense vector search |

In [ ]:
def hyde_retrieve(question: str, k: int = 3) -> list:
    # Step 1: generate a hypothetical answer
    hyp_resp = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": (
            f"Write a short factual paragraph answering: '{question}'. "
            "Some details may be approximate."
        )}],
        options={"temperature": 0.3},
    )
    hyp_doc = hyp_resp["message"]["content"]
    print(f"Hypothetical doc (first 200 chars):\n  {hyp_doc[:200]}\n")

    # Step 2: embed the hypothetical doc
    hyp_vec = embed_model.encode([hyp_doc], normalize_embeddings=True).tolist()

    # Step 3: search with hypothetical embedding
    res = col.query(query_embeddings=hyp_vec, n_results=k)
    return res["documents"][0]

Q = "What type of train operates the SEA-YVR service and how fast can it go?"
chunks = hyde_retrieve(Q)
print("HyDE-retrieved chunks:")
for c in chunks:
    print(f"  - {c}")

## 5 · Production Failure Modes

| Failure Mode | Symptom | Fix |
|-------------|---------|-----|
| **Semantic gap** | Right answer exists but wrong chunks retrieved | Use HyDE or query expansion |
| **Chunk too large** | Relevant sentence diluted in big chunk | Smaller chunks; hierarchical chunking |
| **Lost-in-the-middle** | LLM ignores chunks in the middle of context | Put most relevant chunk first AND last |
| **Context overflow** | Too many chunks; noise drowns signal | Reduce k; use a cross-encoder re-ranker |
| **No answer in corpus** | LLM hallucinates when corpus is silent | Add explicit fallback instruction |
| **Unfaithful generation** | LLM answers from memory not context | Strong grounding instruction in prompt |

## 6 · Key Takeaways

| Concept | One-Liner |
|---------|-----------|
| Embedding | Text → fixed vector; similar meaning → close in space |
| Pooling | Mean pooling outperforms [CLS] on most tasks |
| Contrastive learning | Push similar together, push unrelated apart |
| RAG pipeline | Chunk then embed offline; embed + retrieve at query time |
| HyDE | Embed a hypothetical answer, not the raw question |

**Next:** `03-vector-dbs/notebook.ipynb` — how the vector store finds those chunks in milliseconds.

## 6 · PizzaBot Connection — The RAG Corpus in Practice

> Full system spec: [AIPrimer.md](../AIPrimer.md)

The PizzaBot RAG corpus is a concrete, small-scale instance of every decision in this notebook:

| Decision | PizzaBot choice | Notebook section |
|---|---|---|
| **Embedding model** | `all-MiniLM-L6-v2` (384d) — same model at ingestion and query time | §1 |
| **Chunking strategy** | Per-item for `menu.json` and `allergens.csv`; per-section for `faq.md` | §RAGAndEmbeddings.md §8 |
| **Same-model constraint** | The query `"something light and vegetarian"` uses the same MiniLM weights that embedded the Veggie Feast entry | §2 |
| **Semantic retrieval** | Query `"light vegetarian"` retrieves Veggie Feast even though those exact words don't appear in `menu.json` | §3 |
| **HyDE** | Generates a hypothetical answer (`"A light vegetarian option would be a Margherita or a garden salad pizza..."`) and embeds *that* to retrieve the closest menu item | §4 |
| **Failure mode: lost-in-middle** | Long conversation with 5 retrieved chunks: the allergen chunk in position 3 (middle) is most likely to be ignored by the model | §5 |

**Try it:** replace the ChromaDB documents in §3 with the PizzaBot corpus files from `projects/ai/rag-pipeline/` when that project is completed.
